## Reading content from file

In [10]:
import os

# Your existing code to find files
files = []
for root, dirs, filenames in os.walk(r".\data"):
    for file in filenames:
        file_path = os.path.join(root, file)
        files.append(file_path)

# New code to read and print each file's content
for path in files:
    try:
        # Use 'with' to ensure the file is closed automatically
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
            print(f"--- Content of {path} ---")
            # chunk = create_chunk(content)
            # print("Chunks:", chunk)  # Print the chunks created from the content
            # print(f"Chunk created for {path}: {chunk[0:50]}...")  # Print the first 50 characters of the chunk
            # embeddings = create_embeddings(chunk)
            # print("Embedding 1", embeddings[0])  # Print the first embedding vector
            # print("Embedding -1", embeddings[-1])  
            print(content[0:50])  # Print the first 50 characters to avoid overwhelming output
            print("-" * 30)
    except Exception as e:
        print(f"Could not read {path}: {e}")


--- Content of .\data\security\it security and data privacy.txt ---
# Global IT Security, Data Classification, and Acc
------------------------------
--- Content of .\data\training\learning and tuition.txt ---
# Corporate Policy: Global Learning, Development, 
------------------------------
--- Content of .\data\travel\Fuel and Mileage Policy.txt ---
# Corporate Travel Policy: Personal Vehicle, Fuel,
------------------------------
--- Content of .\data\travel\International Travel.txt ---
# Corporate Travel Policy: International Operation
------------------------------
--- Content of .\data\travel\Travel Policy.txt ---
# Corporate Travel Policy: Standard and Client-Fac
------------------------------
--- Content of .\data\Work policies\code of conduct.txt ---
# Human Resources Policy: Global Code of Conduct a
------------------------------
--- Content of .\data\Work policies\leave_and_absence.txt ---
# Human Resources Policy: Global Leave, Paid Time 
------------------------------
--- Co

In [ ]:
print(files[0])
with open(files[0], 'r', encoding='utf-8') as f:
    content = f.read()
    chunk = create_chunk(content)
    print("Chunks:", chunk)  # Print the chunks created from the content
    embeddings = create_embeddings(chunk)
    print("Embedding 1", embeddings[0])  # Print the first embedding vector
    ingest_into_vector_store(embeddings, chunk)


.\data\security\it security and data privacy.txt
Chunks: ['# Global IT Security, Data Classification, and Acceptable Use Policy\n**Document ID:** SEC-POL-8005-V7\n**Effective Date:** June 1, 2026\n**Last Revised:** April 15, 2026\n**Policy Owner:** Chief Information Security Officer (CISO)\n**Applies To:** All employees, contractors, third-party vendors, and any entity with provisioned access to the corporate network or data infrastructure.\n\n---\n\n## Table of Contents\n1. Security Philosophy and Zero-Trust Architecture\n2. Data Classification Matrix\n3. Acceptable Use of Corporate Systems\n4. Bring Your Own Device (BYOD) and Mobile Management\n5. Password Policies and Authentication\n6. Data Privacy and Handling of PII/Financial Data\n7. Incident Response and Breach Reporting SLAs\n8. Software Procurement and "Shadow IT"\n9. Offboarding and Access Revocation\n\n---', '---\n\n## 1. Security Philosophy and Zero-Trust Architecture\nThe organization operates on a "Zero-Trust" security m

## Chunking

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

def create_chunk(text):
    return text_splitter.split_text(text)

In [15]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
# Initialize the client with your API key
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(
    api_key=API_KEY
)

def create_embeddings(chunks):
    response = client.embeddings.create(
        input=chunks,
        model="text-embedding-3-small"
    )
    embeddings = []
    for i, data in enumerate(response.data):
        embeddings.append({
            "chunk": chunks[i],
            "embedding": data.embedding
        })
    return embeddings



In [18]:
from pinecone import Pinecone, ServerlessSpec
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)

pc.create_index(
    name="rag-index",
    dimension=1536, # Example for OpenAI embeddings
    metric="cosine"
)
index = pc.Index("rag-index")

def ingest_into_vector_store(embeddings, chunks):
    vectors_to_upsert = []
    for i, record in enumerate(embeddings.data):
        vectors_to_upsert.append({
            "id": f"chunk-{i}",
            "values": record.embedding
        })

Exception: The official Pinecone python package has been renamed from `pinecone-client` to `pinecone`. Please remove `pinecone-client` from your project dependencies and add `pinecone` instead. See the README at https://github.com/pinecone-io/pinecone-python-client for more information on using the python SDK.